# Embedding with `SentenceTransformer`

In [ ]:
import pickle
from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("all-MiniLM-L6-v2")
with open("model.pkl", "rb") as file:
    model = pickle.load(file)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
import pickle

# Your data (can be a dict, list, class instance, etc.)
# data_to_save = {"name": "Alice", "age": 30, "scores": [85, 92, 78]}

# Save the object to a file
with open("model.pkl", "wb") as file:
    pickle.dump(model, file)


In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730438)

## Embedding our FAQ Data

In [4]:
from ingest import load_faq_data

documents = load_faq_data()

Combine questions & answers, store them in a list

In [5]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Do batch embedding so our machine can take it easy

In [ ]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

# for i in tqdm(range(0, len(texts), batch_size)):
#     batch = texts[i:i + batch_size]
#     batch_vectors = model.encode(batch)
#     vectors.extend(batch_vectors)

with open("vectors.pkl", "rb") as file:
    vectors = pickle.load(file)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [9]:
import numpy as np
X = np.array(vectors)
X.shape

(1350, 384)

In [10]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [16]:
scores = X.dot(v_query)

# slower alternative
# scores = [v_query.dot(X[i]) for i in range(len(X))]

The highest score is the most similar document

In [13]:
idx = np.argmax(scores)
idx, scores[idx]

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

To get the top 5 results, we can use `argsort`

In [14]:
top5 = np.argsort(scores)[-5:]

# sort from the highest
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

Alternatively, in one line:

In [20]:
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [17]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

Getting the real document:

In [21]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 